# Lab 9: Scale Space and Multi-Scale Feature Detection

**Computer Vision Course**

Building on Labs 7 and 8, today you'll discover why **scale matters** in computer vision and how to detect features at multiple scales.

**What you'll learn:**
- The "Goldilocks problem" in computer vision
- Creating scale-space representations (Gaussian pyramids)
- Difference of Gaussians (DoG) as efficient LoG approximation
- Multi-scale edge detection
- Unsharp masking for image enhancement

**What you'll build:**
- A Gaussian pyramid
- Multi-scale feature detector
- Unsharp mask filter

**Why this matters:**
This lab bridges classical computer vision and deep learning. The scale-space concepts you'll learn today are **exactly what CNNs do automatically** in their hierarchical layers!

**Connection to previous labs:**
- Lab 7: Learned convolution and Gaussian smoothing
- Lab 8: Built edge detectors (Sobel, Laplacian, LoG)
- Lab 9 (today): Apply edge detection across **multiple scales**
- Lab 10 (next week): See how CNNs learn multi-scale features automatically!

## Setup

In [ ]:
"""
Computer Vision Course - Lab 9: Scale Space

This cell sets up the environment.
Works automatically for both local and Google Colab!
"""

import os
import sys

# Detect environment
IN_COLAB = 'google.colab' in sys.modules

print("=" * 60)
print("Computer Vision - Lab 9: Scale Space")
print("=" * 60)

if IN_COLAB:
    print("\n🔵 Running on Google Colab")
    print("-" * 60)
    
    if not os.path.exists('computer-vision'):
        print("📥 Cloning repository...")
        !git clone https://github.com/mjck/computer-vision.git
        print("✓ Repository cloned successfully")
    else:
        !git -C computer-vision pull
        print("✓ Repository updated successfully")
    
    %cd computer-vision/labs/lab09_scale_space
    print(f"✓ Current directory: {os.getcwd()}")
    
    sys.path.insert(0, '/content/computer-vision')
    print("✓ Python path configured")
    
    print("-" * 60)
    print("🟢 Colab setup complete!\n")
    
else:
    print("\n🟢 Running locally")
    print("-" * 60)
    print(f"✓ Current directory: {os.getcwd()}")
    
    repo_root = os.path.abspath('../..')
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    print(f"✓ Repository root: {repo_root}")
    
    print("-" * 60)
    print("🟢 Local setup complete!\n")

print("=" * 60)
print("✅ Environment ready!")
print("=" * 60)

## Import Libraries

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy import signal

# Import course utilities
try:
    from sdx import cv_imread, cv_imshow, cv_grayread
    print("✓ sdx module loaded")
except ImportError as e:
    print(f"❌ Could not import sdx: {e}")
    raise

print("✓ All imports successful")

## Convenience Functions

We'll reuse the convolution and normalization functions from Lab 8.

In [ ]:
def convolve(input, kernel):
    """
    Apply 2D convolution using correlation (no kernel flip).
    """
    return signal.correlate2d(input, kernel, mode='valid')

def normalize(output):
    """
    Normalize an array to [0, 255] range.
    """
    min_val = output.min()
    max_val = output.max()
    
    if max_val == min_val:
        return np.full_like(output, 127.5)
    
    return 255 * (output - min_val) / (max_val - min_val)

def gaussian_kernel(size, sigma):
    """
    Create a Gaussian kernel.
    """
    radius = size // 2
    x = np.arange(-radius, radius + 1)
    y = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(x, y)
    kernel = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    kernel = kernel / kernel.sum()  # Normalize
    return kernel

print("✓ Utility functions defined")

## Load Test Image

Same images as Labs 7-8!

In [ ]:
# Choose your source
SOURCE = 'smash'  # Options: 'insper', 'informatica', 'harvard', 'atletica', 'smash', 'consulting'

# Load clean image
input_image = cv_grayread(f'{SOURCE}.png', asfloat=True)

print(f"Image: {SOURCE}")
print(f"Shape: {input_image.shape}")
print(f"Range: [{input_image.min():.3f}, {input_image.max():.3f}]")

cv_imshow(input_image)

---
## The Goldilocks Problem

**The challenge:** Edge detection depends on the **scale** at which we look!

From the slides, you saw three cases:
- **"Too small"** — Edge detector misses large structures
- **"Just right"** — Edge detector matches feature scale
- **"Too large"** — Edge detector misses fine details

Let's see this in action!

### Demonstrate Scale Sensitivity

We'll apply Laplacian edge detection at three different scales (blur levels).

In [ ]:
# Laplacian kernel (same as Lab 8)
L = np.array([
    [ 0,  1,  0],
    [ 1, -4,  1],
    [ 0,  1,  0],
])

# Three different Gaussian blur levels
sigma_small = 0.5   # "Too small" — minimal smoothing
sigma_medium = 1.5  # "Just right"
sigma_large = 4.0   # "Too large" — heavy smoothing

# Create Gaussian kernels
G_small = gaussian_kernel(5, sigma_small)
G_medium = gaussian_kernel(9, sigma_medium)
G_large = gaussian_kernel(17, sigma_large)

# Smooth at each scale
smooth_small = convolve(input_image, G_small)
smooth_medium = convolve(input_image, G_medium)
smooth_large = convolve(input_image, G_large)

# Apply Laplacian to each
edges_small = convolve(smooth_small, L)
edges_medium = convolve(smooth_medium, L)
edges_large = convolve(smooth_large, L)

print("Laplacian edge detection at three scales:")
print(f"\nSmall scale (σ={sigma_small}):")
cv_imshow(normalize(np.abs(edges_small)))
print("→ Picks up fine details, lots of noise")

print(f"\nMedium scale (σ={sigma_medium}):")
cv_imshow(normalize(np.abs(edges_medium)))
print("→ Balanced: major edges visible, less noise")

print(f"\nLarge scale (σ={sigma_large}):")
cv_imshow(normalize(np.abs(edges_large)))
print("→ Only large structures, fine details lost")

print("\n🤔 Different scales capture different features!")

### 💡 Key Insight

**There is no single "right" scale!**

Different features appear at different scales:
- **Fine details** (textures, small edges) → small σ
- **Medium structures** (object boundaries) → medium σ
- **Large structures** (overall shape, layout) → large σ

**Solution:** Analyze the image at **multiple scales simultaneously**!

---
## Part 1: Creating a Scale-Space Pyramid

**The idea:** Create a stack of progressively blurred versions of the same image.

**From the slides:** S_σ, S_2σ, S_3σ, S_4σ

Each level represents the image at a different scale!

### Build Gaussian Pyramid

A **Gaussian pyramid** is a sequence of increasingly blurred images.

In [ ]:
def build_gaussian_pyramid(image, num_levels=4, sigma_base=1.0):
    """
    Build a Gaussian pyramid.
    
    Args:
        image: Input image
        num_levels: Number of pyramid levels
        sigma_base: Base sigma (will be multiplied for each level)
    
    Returns:
        List of smoothed images [S_σ, S_2σ, S_3σ, S_4σ]
    """
    pyramid = []
    
    for level in range(num_levels):
        # Sigma increases with level
        sigma = sigma_base * (level + 1)
        
        # Create Gaussian kernel
        # Kernel size should scale with sigma
        kernel_size = int(6 * sigma) + 1  # Rule of thumb
        if kernel_size % 2 == 0:  # Must be odd
            kernel_size += 1
        
        G = gaussian_kernel(kernel_size, sigma)
        
        # Smooth image
        smoothed = convolve(image, G)
        pyramid.append(smoothed)
        
        print(f"Level {level}: σ={sigma:.1f}, kernel size={kernel_size}x{kernel_size}")
    
    return pyramid

# Build pyramid
pyramid = build_gaussian_pyramid(input_image, num_levels=4, sigma_base=1.0)

print(f"\n✓ Created pyramid with {len(pyramid)} levels")

### Visualize Gaussian Pyramid

In [ ]:
print("Gaussian Pyramid Visualization:\n")

for i, level in enumerate(pyramid):
    sigma = 1.0 * (i + 1)
    cv_imshow(level)
    print(f"Level {i}: S_{{{sigma:.0f}σ}} — shape {level.shape}")

print("\n✓ Notice how details gradually disappear!")

---
## ✏️ Exercise 1: Create Your Own Pyramid

**Task:** Build a Gaussian pyramid with **different parameters**.

**Instructions:**
1. Choose a different source image
2. Create a pyramid with 5 levels
3. Use sigma_base = 0.8
4. Display all 5 levels
5. Describe what you observe

In [ ]:
# ── Your code here ──────────────────────────────────────────────────────────

# Step 1: Choose a source (change SOURCE variable)
# SOURCE = '...'  # Pick: insper, informatica, harvard, atletica, smash, consulting

# Step 2: Load image
# image = cv_grayread(f'{SOURCE}.png', asfloat=True)

# Step 3: Build pyramid with 5 levels, sigma_base=0.8
# my_pyramid = build_gaussian_pyramid(image, num_levels=5, sigma_base=0.8)

# Step 4: Display all levels
# for i, level in enumerate(my_pyramid):
#     ...

# Step 5: Write your observations

# ────────────────────────────────────────────────────────────────────────────

print("Complete Exercise 1 above")

---
## Part 2: Difference of Gaussians (DoG)

**Key formula from slides:**

```
S_kσ - S_σ ≈ (k-1) · σ² · ∇²S_σ
```

**What this means:** The **difference** between two Gaussian-smoothed images approximates the **Laplacian** (second derivative)!

**Why this matters:**
- **Efficient:** We already have the Gaussian pyramid!
- **No need to explicitly compute Laplacian**
- **Foundation of SIFT feature detection**

**The insight:** Scale space gives you Laplacians "for free"!

### Compute Difference of Gaussians

In [ ]:
def compute_dog_pyramid(gaussian_pyramid):
    """
    Compute Difference of Gaussians between adjacent pyramid levels.
    
    Args:
        gaussian_pyramid: List of Gaussian-smoothed images
    
    Returns:
        List of DoG images
    """
    dog_pyramid = []
    
    for i in range(len(gaussian_pyramid) - 1):
        # Difference between adjacent levels
        dog = gaussian_pyramid[i+1] - gaussian_pyramid[i]
        dog_pyramid.append(dog)
        
        sigma1 = 1.0 * (i + 1)
        sigma2 = 1.0 * (i + 2)
        print(f"DoG {i}: S_{{{sigma2:.0f}σ}} - S_{{{sigma1:.0f}σ}}")
    
    return dog_pyramid

# Compute DoG pyramid
dog_pyramid = compute_dog_pyramid(pyramid)

print(f"\n✓ Created {len(dog_pyramid)} DoG images")

### Visualize DoG Pyramid

DoG images show **edges at different scales**.

In [ ]:
print("Difference of Gaussians Pyramid:\n")

for i, dog in enumerate(dog_pyramid):
    cv_imshow(normalize(np.abs(dog)))
    sigma1 = 1.0 * (i + 1)
    sigma2 = 1.0 * (i + 2)
    print(f"DoG {i}: S_{{{sigma2:.0f}σ}} - S_{{{sigma1:.0f}σ}}")

print("\n🎯 Each DoG captures edges at a specific scale!")

### Compare DoG vs LoG

Let's verify the approximation: **DoG ≈ LoG**

We'll compare DoG to explicitly computing LoG at the same scale.

In [ ]:
# Take the first DoG (between S_σ and S_2σ)
dog_example = dog_pyramid[0]

# Compute explicit LoG at intermediate scale (σ ≈ 1.5)
sigma_log = 1.5
kernel_size = int(6 * sigma_log) + 1
if kernel_size % 2 == 0:
    kernel_size += 1

# Create LoG kernel by convolving Gaussian with Laplacian
G_log = gaussian_kernel(kernel_size, sigma_log)
LoG_kernel = convolve(G_log, L)

# Apply LoG
log_result = convolve(input_image, LoG_kernel)

print("Comparison: DoG vs LoG\n")

print("DoG (S_2σ - S_σ):")
cv_imshow(normalize(np.abs(dog_example)))

print("\nLoG (Gaussian σ=1.5 + Laplacian):")
cv_imshow(normalize(np.abs(log_result)))

print("\n✓ Very similar! DoG is an efficient approximation of LoG")

---
## ✏️ Exercise 2: Analyze DoG at Different Scales

**Task:** Compare what features are captured by DoG at different pyramid levels.

**Instructions:**
1. Look at DoG level 0 (small scale differences)
2. Look at DoG level 2 (large scale differences)
3. Identify what types of features each one captures
4. Answer: Which level shows **fine details**? Which shows **coarse structure**?

**Hint:** Fine details appear at smaller scales, coarse structure at larger scales.

In [ ]:
# ── Your code here ──────────────────────────────────────────────────────────

# Display DoG at level 0 (fine details)
# cv_imshow(normalize(np.abs(dog_pyramid[0])))
# print("DoG level 0 captures: ...")

# Display DoG at level 2 (coarse structure)
# cv_imshow(normalize(np.abs(dog_pyramid[2])))
# print("DoG level 2 captures: ...")

# Answer the questions:
# Which level shows fine details?
# Which level shows coarse structure?

# ────────────────────────────────────────────────────────────────────────────

print("Complete Exercise 2 above")

---
## Part 3: Multi-Scale Edge Detection

**Approach 1 from slides:** Detect edges at all scales, then **consolidate** the results.

**Consolidation strategies:**
- **Maximum:** Take strongest edge at each pixel across all scales
- **Minimum:** Take weakest edge
- **Mean:** Average across scales

We'll use the **maximum** approach — this gives us edges that are strong at **any** scale!

### Apply Laplacian at All Scales

In [ ]:
# Apply Laplacian to each level of Gaussian pyramid
laplacian_pyramid = []

for i, smooth_img in enumerate(pyramid):
    edges = convolve(smooth_img, L)
    laplacian_pyramid.append(edges)
    sigma = 1.0 * (i + 1)
    print(f"Laplacian at σ={sigma:.0f}: shape {edges.shape}")

print(f"\n✓ Created {len(laplacian_pyramid)} Laplacian edge maps")

### Consolidate: Maximum Across Scales

Take the **maximum absolute value** at each pixel position across all scales.

**Challenge:** Different pyramid levels have different sizes (due to border handling).
We'll resize them to match for comparison.

In [ ]:
# Find the smallest size (last pyramid level has smallest size)
min_shape = laplacian_pyramid[-1].shape

# Resize all to match smallest size
laplacian_resized = []
for i, lap in enumerate(laplacian_pyramid):
    # Use cv2.resize to match smallest size
    # Convert to uint8 for resize, then back to float
    lap_norm = normalize(np.abs(lap)).astype(np.uint8)
    lap_resized = cv2.resize(lap_norm, (min_shape[1], min_shape[0]), interpolation=cv2.INTER_LINEAR)
    laplacian_resized.append(lap_resized.astype(float))

# Stack into 3D array (scales × height × width)
laplacian_stack = np.stack(laplacian_resized, axis=0)

# Take maximum across scale axis
max_across_scales = np.max(laplacian_stack, axis=0)

print("Multi-scale edge detection:\n")

print("Individual scales:")
for i, lap in enumerate(laplacian_resized):
    cv_imshow(lap)
    sigma = 1.0 * (i + 1)
    print(f"Scale σ={sigma:.0f}")

print("\nMaximum across all scales:")
cv_imshow(max_across_scales)

print("\n🎯 Captures edges that are strong at ANY scale!")

---
## ✏️ Exercise 3: Multi-Scale Consolidation (4 points)

**Task:** Compare different consolidation strategies.

**Instructions:**
1. Compute **maximum** across scales (already done above)
2. Compute **minimum** across scales
3. Compute **mean** across scales
4. Display all three results
5. Answer: Which strategy gives the **most complete edge map**? Why?

In [ ]:
# ── Your code here ──────────────────────────────────────────────────────────

# Compute minimum across scales
# min_across_scales = ...

# Compute mean across scales
# mean_across_scales = ...

# Display all three
# cv_imshow(max_across_scales)
# print("Maximum")

# cv_imshow(min_across_scales)
# print("Minimum")

# cv_imshow(mean_across_scales)
# print("Mean")

# Answer: Which gives the most complete edge map? Why?
# ...

# ────────────────────────────────────────────────────────────────────────────

print("Complete Exercise 3 above")

---
## Part 4: Unsharp Masking

**From the slides:** A technique to **enhance details** in an image.

**The idea:**
```
S - U = Details only
P - k·U = Enhanced image (k controls strength)
```

Where:
- S = Smoothed image
- U = Unsharp mask (smoothed - original)
- P = Original image
- k = Enhancement strength

**Connection to Laplacian:** This is actually **related to the Laplacian**!
```
P - (S - P) = P - S + P = 2P - S
```

The slides showed: P - 2U, P - 3U, P - 4U

### Implement Unsharp Masking

In [ ]:
def unsharp_mask(image, sigma=2.0, strength=1.0):
    """
    Apply unsharp masking to enhance details.
    
    Args:
        image: Input image
        sigma: Blur amount for smoothing
        strength: Enhancement strength (k parameter)
    
    Returns:
        Enhanced image
    """
    # Create Gaussian kernel
    kernel_size = int(6 * sigma) + 1
    if kernel_size % 2 == 0:
        kernel_size += 1
    G = gaussian_kernel(kernel_size, sigma)
    
    # Smooth the image
    smoothed = convolve(image, G)
    
    # Compute unsharp mask: smoothed - original
    # Need to crop original to match smoothed size
    radius = kernel_size // 2
    image_cropped = image[radius:-radius, radius:-radius]
    
    mask = smoothed - image_cropped
    
    # Enhanced = original - k * mask
    enhanced = image_cropped - strength * mask
    
    # Clip to valid range
    enhanced = np.clip(enhanced, 0, 255)
    
    return enhanced

# Apply with different strengths
print("Unsharp masking with different strengths:\n")

strengths = [0, 1, 2, 3, 4]
for k in strengths:
    if k == 0:
        # Just show original (cropped to match)
        kernel_size = int(6 * 2.0) + 1
        if kernel_size % 2 == 0:
            kernel_size += 1
        radius = kernel_size // 2
        result = input_image[radius:-radius, radius:-radius]
        print("Original (no enhancement):")
    else:
        result = unsharp_mask(input_image, sigma=2.0, strength=k)
        print(f"Strength k={k}:")
    
    cv_imshow(result)

print("\n✓ Higher k = stronger detail enhancement!")
print("✓ Too high k can create halos and artifacts")

### Visualize the Unsharp Mask Itself

Let's see what the "mask" actually looks like:

In [ ]:
# Create smoothed version
sigma = 2.0
kernel_size = int(6 * sigma) + 1
if kernel_size % 2 == 0:
    kernel_size += 1
G = gaussian_kernel(kernel_size, sigma)
smoothed = convolve(input_image, G)

# Crop original
radius = kernel_size // 2
image_cropped = input_image[radius:-radius, radius:-radius]

# Compute mask
mask = smoothed - image_cropped

print("Understanding the unsharp mask:\n")

print("Original (cropped):")
cv_imshow(image_cropped)

print("Smoothed:")
cv_imshow(smoothed)

print("Mask (smoothed - original):")
cv_imshow(normalize(mask))
print("→ Shows what was removed by smoothing (the details!)")

print("\nEnhanced (original - 2×mask):")
enhanced = image_cropped - 2.0 * mask
enhanced = np.clip(enhanced, 0, 255)
cv_imshow(enhanced)
print("→ Subtracting the mask adds back the details!")

---
## 🧠 Connection to Convolutional Neural Networks

**Everything you learned today is what CNNs do automatically!**

### Scale-Space → CNN Hierarchy

**Today you learned:**
1. ✅ Features exist at multiple scales
2. ✅ Need to analyze images at different resolutions
3. ✅ Combine information across scales
4. ✅ Edge-like filters at each scale

**In CNNs (next week):**
1. ✅ Early layers detect edges (like your Sobel/Laplacian)
2. ✅ Middle layers detect patterns at medium scales
3. ✅ Deep layers detect high-level features (objects, faces)
4. ✅ **Pooling layers** create scale hierarchies (like pyramids!)

### The Key Difference

**Today (handcrafted):**
- You designed filters manually
- You chose σ values
- You decided consolidation strategy

**CNNs (learned):**
- Network **learns optimal filters** from data
- Network **learns how to combine scales**
- Network **learns what features matter**

**But the underlying principles are the same!**

### Visualization of CNN Filters

When researchers visualize what CNNs learn in early layers, they see:
- **Layer 1:** Edge detectors (horizontal, vertical, diagonal) — **like Sobel!**
- **Layer 2:** Corners, textures — **like DoG!**
- **Layer 3:** Simple shapes
- **Layer 4+:** Complex patterns, object parts

**Your scale-space pyramid ≈ Early CNN layers!**

The manual work you did today helps you understand what CNNs do "under the hood"!

### ✏️ Final Reflection (+1 point)

Answer these questions:

1. **Why does scale matter in computer vision?**

2. **What is the advantage of DoG over explicitly computing LoG?**

3. **How does today's content connect to CNNs?**

4. **When might you want to use multiple scales in a real application?**

In [ ]:
# Your reflection:

# 1. Why does scale matter?
#    ...

# 2. Advantage of DoG?
#    ...

# 3. Connection to CNNs?
#    ...

# 4. Real application example?
#    ...
